In [3]:
import pandas as pd
import numpy as np
import os

os.makedirs('data', exist_ok=True)

# Load data
df = pd.read_csv('anaconda_projects/18244f02-dc80-4393-afa7-b7e805dc0cfa/WA_Fn-UseC_-Telco-Customer-Churn.csv')

# --- Basic exploration ---
print("Shape:", df.shape)
print("\nInfo:")
print(df.info())
print("\nMissing values:\n", df.isnull().sum())
print("\nChurn distribution:\n", df['Churn'].value_counts())

# --- Fix TotalCharges (usually stored as object/string with blanks) ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("\nNulls in TotalCharges after conversion:", df['TotalCharges'].isnull().sum())

# Fill missing TotalCharges with tenure * MonthlyCharges (reasonable estimate)
df['TotalCharges'] = df['TotalCharges'].fillna(df['tenure'] * df['MonthlyCharges'])

# --- Drop customerID (not useful for model) but keep a copy for reference ---
customer_ids = df['customerID']
df.drop('customerID', axis=1, inplace=True)

# --- Feature Engineering ---

# 1. Tenure groups
def tenure_group(tenure):
    if tenure <= 12:
        return '0-12'
    elif tenure <= 24:
        return '13-24'
    elif tenure <= 48:
        return '25-48'
    else:
        return '48+'

df['tenure_group'] = df['tenure'].apply(tenure_group)

# 2. Charges ratio (avoid divide-by-zero)
df['charges_ratio'] = df['MonthlyCharges'] / df['TotalCharges'].replace(0, np.nan)
df['charges_ratio'] = df['charges_ratio'].fillna(0)

# 3. Service count — count how many services customer has opted into
service_cols = ['PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
                 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

def count_services(row):
    count = 0
    for col in service_cols:
        val = row[col]
        if val not in ['No', 'No internet service', 'No phone service']:
            count += 1
    return count

df['service_count'] = df.apply(count_services, axis=1)

# --- Encode target variable ---
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# --- One-Hot Encode categorical columns (excluding target) ---
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print("\nCategorical columns to encode:", categorical_cols)

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# --- Save cleaned data (with customerID added back for reference) ---
df_encoded['customerID'] = customer_ids.values
df_encoded.to_csv('data/cleaned_churn_data.csv', index=False)

print("\n✅ Cleaned data saved to data/cleaned_churn_data.csv")
print("Final shape:", df_encoded.shape)
print("\nColumns:\n", df_encoded.columns.tolist())

Shape: (7043, 21)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling 